In [2]:
import sys

assert sys.version_info >= (3,10)

In [3]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [4]:
import matplotlib.pyplot as  plt

plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('font', size=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [5]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "07_Euler_Bernoulli_Beam_Equation_with_Higher_order"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [6]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [7]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Exact solution:
Got this exact solution by integrating the bernoulli equation and substituting the boundary conditions for cantilever beam.

In [8]:
def exact_solution(x):
    return -(1/24) * x ** 4 + (1/6) * x ** 3 - (1/4) * x ** 2

Creating the geometry for the beam of unit length

In [9]:
geom =  dde.geometry.Interval(0, 1)


Creating a Partial differential equation for external distributed load with 1KN/m

In [10]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    dy_xxxx = dde.grad.hessian(dy_xx, x, i=0, j=0)
    return dy_xxxx + 1

creating the boundary conditions at x = 0

In [11]:
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], 0)

bc_u_0 = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)
bc_du_0 = dde.icbc.NeumannBC(geom, lambda x:0, boundary_left)

Creating the Boundary Conditions at x = 1

In [12]:
def boundary_right(x, on_boundary):
    return on_boundary and np.isclose(x[0], 1)

def boundary_second_derivative(x, y, X):
    return dde.grad.hessian(y, x, i=0, j=0)

bc_obc1 = dde.icbc.OperatorBC(
    geom, boundary_second_derivative,
    boundary_right
)

def boundary_third_derivative(x, y, X):
    d2y_dx2 = dde.grad.hessian(y, x, i=0, j=0)
    return dde.grad.jacobian(d2y_dx2, x, i=0, j=0)
bc_obc2 = dde.icbc.OperatorBC(
    geom, boundary_third_derivative, boundary_right
)

Creating training data points

In [13]:
observe_x  = np.linspace(0, 1, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise = 0.05 * observe_y.std() * np.random.randn(35, 1)
observe_y = noise + observe_y
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)

Combining all the data

In [14]:
data = dde.data.PDE(
    geom, pde, [bc_u_0, bc_du_0, bc_obc1, bc_obc2, observe],
    num_domain=3000, num_boundary=200, num_test=500
)

Build a Neural Network and Model

In [15]:
net = dde.nn.FNN([1, 128, 64, 32, 1], "tanh", "Glorot uniform")
model = dde.Model(data, net)

Set loss Weights

In [16]:
loss_weights = [1.0, 20.0, 20.0, 20.0, 20.0, 500.0]

Training the model with
    1. Adam Optimizer
    2. L-BFGS Optimizer

In [17]:
print("\nStage 1: Adam Optimizer")
model.compile(
    "adam", lr=0.001, loss_weights=loss_weights,
    decay=("inverse time", 4000, 0.1)
)
losshistory, train_state = model.train(iterations = 2000, display_every=500)

dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights=loss_weights)
losshistory, train_state = model.train(display_every=50)


Stage 1: Adam Optimizer
Compiling model...
'compile' took 1.912530 s

Training model...



/home/ziaur/ziazh/lib/python3.14/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step      Train loss                                                      Test loss                                                       Test metric
0         [9.23e-01, 0.00e+00, 1.76e+00, 8.57e-02, 3.21e-02, 5.48e+00]    [9.22e-01, 0.00e+00, 1.76e+00, 8.57e-02, 3.21e-02, 5.48e+00]    []  
500       [3.78e-02, 1.57e-06, 1.47e-03, 7.79e-04, 1.19e-03, 2.39e-03]    [3.07e-02, 1.57e-06, 1.47e-03, 7.79e-04, 1.19e-03, 2.39e-03]    []  
1000      [1.32e-02, 4.28e-08, 1.85e-04, 9.45e-05, 1.19e-04, 1.60e-03]    [1.03e-02, 4.28e-08, 1.85e-04, 9.45e-05, 1.19e-04, 1.60e-03]    []  
1500      [5.37e-03, 2.06e-06, 1.15e-04, 5.31e-05, 3.74e-05, 1.76e-03]    [4.00e-03, 2.06e-06, 1.15e-04, 5.31e-05, 3.74e-05, 1.76e-03]    []  
2000      [2.11e-03, 3.09e-08, 1.47e-05, 1.38e-05, 4.45e-06, 1.62e-03]    [1.56e-03, 3.09e-08, 1.47e-05, 1.38e-05, 4.45e-06, 1.62e-03]    []  

Best model at step 2000:
  train loss: 3.76e-03
  test loss: 3.21e-03
  test metric: []

'train' took 512.983625 s

Compiling model...

Evaluating the Model

Creating test points

In [22]:
test_points = 15
x_test = np.linspace(0, 1, test_points).reshape(-1, 1)
y_pred = model.predict(x_test)
y_exact = exact_solution(x_test)
noise = 0.05 * y_exact.std() * np.random.randn(15, 1)   # 5% noise
y_test = y_exact + noise

E = 210e9     # Young's Modulus (Pa)
I = 1.667e-5  # Moment of Inertia (m^4)
EI = E * I


def calculate_beam_physics(x, y):
    slope_theta = dde.grad.jacobian(y, x, i=0, j=0)
    d2w_dx2 = dde.grad.jacobian(slope_theta, x, i=0, j=0)
    
    # Calculate the true structural Bending Moment
    bending_moment = EI * d2w_dx2
    shear_force = dde.grad.jacobian(bending_moment, x, i=0, j=0)
    
    return [y, slope_theta, bending_moment, shear_force]

results = model.predict(x_test, operator=calculate_beam_physics)

# 5. Extract your clean NumPy arrays
w_pred   = results[0]
slope_np = results[1]
moment_M = results[2]
shear_V  = results[3]

x_vals      = x_test.flatten()
w_vals      = results[0].flatten()
slope_vals  = results[1].flatten()
moment_vals = results[2].flatten()
shear_vals  = results[3].flatten()

print(f"{'X Coord (m)':<15} {'Deflection (m)':<18} {'Slope (rad)':<15} {'Moment (N·m)':<15} {'Shear Force (N)':<15}")
print("-" * 80)

for x, w, theta, M, V in zip(x_vals, w_vals, slope_vals, moment_vals, shear_vals):
    print(f"{x:<15.3f} {w:<18.5e} {theta:<15.5e} {M:<15.2f} {V:<15.2f}")

X Coord (m)     Deflection (m)     Slope (rad)     Moment (N·m)    Shear Force (N)
--------------------------------------------------------------------------------
0.000           5.63694e-05        -7.54721e-04    -1749336.37     3501255.47     
0.071           -1.21265e-03       -3.39576e-02    -1508173.04     3251349.83     
0.143           -4.68190e-03       -6.24219e-02    -1284864.87     3001218.79     
0.214           -1.00259e-02       -8.65122e-02    -1079430.98     2750943.56     
0.286           -1.69453e-02       -1.06593e-01    -891868.24      2500848.64     
0.357           -2.51667e-02       -1.23029e-01    -722164.30      2250855.96     
0.429           -3.44428e-02       -1.36185e-01    -570321.47      2000698.86     
0.500           -4.45523e-02       -1.46425e-01    -436356.51      1750286.23     
0.571           -5.52999e-02       -1.54113e-01    -320282.22      1499808.76     
0.643           -6.65164e-02       -1.59616e-01    -222092.56      1249558.17     
0.714 